In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import sys

sys.path.append("..")

import itertools
import json
from functools import partial
from typing import Any

import seaborn as sns
from datasets import disable_caching
from datasets.utils.logging import disable_progress_bar
from hydra import compose, initialize
from scipy.stats import pearsonr, spearmanr

from hallucinations_kg.data.postprocessing import (
    add_allowed_nodes_and_relationships,
    parse_graph,
    postprocess_ents_rels,
)
from hallucinations_kg.data.utils import get_processed_dataset
from hallucinations_kg.metrics import auc_pr
from hallucinations_kg.models.baselines.selfcheckgpt import SelfCheckGPTPredictor
from hallucinations_kg.prediction.prediction import (
    add_sentence_predictor_results,
)

disable_caching()
disable_progress_bar()

sns.set_style("whitegrid")
sns.set_palette("colorblind")

In [11]:
def parse_params(params: str) -> dict[str, Any]:
    return json.loads(params.replace("'", '"'))

In [12]:
with initialize(version_base="1.3", config_path=str("config")):
    cfg = compose(config_name="predict", overrides=["llm=api/Llama-3.1-70B-Instruct"])

dataset = get_processed_dataset(cfg.processed_dataset)

In [13]:
for prompt_name in ["selfcheckgpt-original", "selfcheckgpt-enhanced"]:
    with initialize(version_base="1.3", config_path=str("config")):
        cfg = compose(
            config_name="predict",
            overrides=["llm=api/Llama-3.1-70B-Instruct", f"selfcheckgpt_prompt_name={prompt_name}"],
        )

    pred_dataset = get_processed_dataset(cfg.processed_dataset)

    restrictions_column = cfg.processed_dataset.original_dataset.response_column
    response_sentences_column = cfg.processed_dataset.original_dataset.response_sentences_column

    pred_dataset = pred_dataset.map(
        partial(
            add_allowed_nodes_and_relationships,
            restrictions_column=restrictions_column,
            response_sentences_column=response_sentences_column,
        )
    )
    pred_dataset = pred_dataset.map(partial(parse_graph, restrict=True))
    pred_dataset = pred_dataset.map(postprocess_ents_rels)

    PREDICTOR_AGGREGATION_FUNCTIONS = [(SelfCheckGPTPredictor, None)]

    for name, ds in pred_dataset.items():
        y_true = list(itertools.chain.from_iterable(ds["binary_annotation"]))
        ds = ds.map(
            partial(
                add_sentence_predictor_results,
                y_true=y_true,
                predictor_aggregation_functions=PREDICTOR_AGGREGATION_FUNCTIONS,
            )
        )

        for col in ds.column_names:
            if col.startswith("sent_score_") or col.startswith("doc_score_"):
                params = parse_params(col.split("__")[-1])
                if (
                    "sentence_predictor" in params
                    and params["sentence_predictor"] == "SelfCheckGPT-reproduced"
                ):
                    params["prompt_name"] = prompt_name
                    new_col = f"{col.split('__')[0]}__{json.dumps(params)}"
                    dataset[name] = dataset[name].add_column(new_col, ds[col])

In [14]:
df = dataset["evaluation"].to_pandas()

In [15]:
SENTENCE_SCORE_COLUMNS = [col for col in df.columns if col.startswith("sent_score_")]
COLUMNS_TO_EXPLODE = [
    "gpt3_sentences",
    "binary_annotation",
] + SENTENCE_SCORE_COLUMNS

sentence_df = df.explode(COLUMNS_TO_EXPLODE)

In [16]:
import pandas as pd
from sklearn.metrics import precision_recall_curve

results = []
for col in SENTENCE_SCORE_COLUMNS:
    y_pred = sentence_df[sentence_df[col].notna()][col]
    y_true = sentence_df[sentence_df[col].notna()].binary_annotation.to_numpy(dtype=float)

    auc_pr_hallucination = auc_pr(y_true, y_pred) * 100
    prec, recall, thresholds = precision_recall_curve(y_true, y_pred)
    results.append(
        {
            "col": col,
            **parse_params(col.split("__")[-1]),
            "auc_hallucination": auc_pr_hallucination,
            "Precision": prec,
            "Recall": recall,
            "Thresholds": thresholds,
        }
    )

results = pd.DataFrame(results)

In [17]:
mask = results.sentence_predictor.str.contains("SelfCheckGPT")


def get_method_name(row: pd.Series) -> str:
    if row["sentence_predictor"] == "SelfCheckGPT-reproduced":
        return "SelfCheckGPT-reproduced"
    else:
        raise ValueError()


results["Method"] = results.apply(get_method_name, axis=1)
to_report = results[mask][["Method", "prompt_name", "auc_hallucination"]]

to_report = to_report.rename(
    columns={
        "auc_hallucination": "AUC-PR-Hallucination",
        "prompt_name": "Prompt",
    }
)

to_report = to_report.sort_values(by="AUC-PR-Hallucination", ascending=False)
to_report = to_report.rename(columns={"AUC-PR-Hallucination": "AUC-PR"})
display(to_report)

print(to_report.to_latex(index=False, float_format="%.2f"))

,Method,Prompt,AUC-PR
0,SelfCheckGPT-reproduced,selfcheckgpt-original,93.603953
1,SelfCheckGPT-reproduced,selfcheckgpt-enhanced,93.379924


\begin{tabular}{llr}
\toprule
Method & Prompt & AUC-PR \\
\midrule
SelfCheckGPT-reproduced & selfcheckgpt-original & 93.60 \\
SelfCheckGPT-reproduced & selfcheckgpt-enhanced & 93.38 \\
\bottomrule
\end{tabular}



### Document level

In [18]:
DOCUMENT_SCORE_COLUMNS = [col for col in df.columns if col.startswith("doc_score_")]

results = []

for filter_not_na in [True]:
    cols = DOCUMENT_SCORE_COLUMNS
    for col in cols:
        params = parse_params(col.split("__")[-1])
        if "filter_not_na" in params and params["filter_not_na"] != filter_not_na:
            continue
        y_pred = df[col].to_numpy(dtype=float)
        y_true = df.document_annotation.to_numpy(dtype=float)
        results.append(
            {
                "filter_not_na": filter_not_na,
                "col": col,
                **params,
                "Pearson": pearsonr(y_true, y_pred)[0] * 100,
                "Spearman": spearmanr(y_true, y_pred)[0] * 100,
            }
        )

docs_results = pd.DataFrame(results)

In [19]:
mask = docs_results.sentence_predictor.str.contains("SelfCheckGPT")
docs_results["Method"] = docs_results.apply(get_method_name, axis=1)
to_report = docs_results[
    [
        "Method",
        "prompt_name",
        "Pearson",
        "Spearman",
    ]
]
to_report = to_report[docs_results.filter_not_na & mask].sort_values(by="Pearson", ascending=False)
to_report = to_report.rename(
    columns={
        "auc": "AUC-PR",
        "prompt_name": "Prompt",
    }
)
to_report = to_report.sort_values(by="Pearson", ascending=False)
display(to_report)
print(to_report.to_latex(index=False, float_format="%.2f"))

,Method,Prompt,Pearson,Spearman
0,SelfCheckGPT-reproduced,selfcheckgpt-original,83.120336,78.648982
1,SelfCheckGPT-reproduced,selfcheckgpt-enhanced,82.357701,76.705938


\begin{tabular}{llrr}
\toprule
Method & Prompt & Pearson & Spearman \\
\midrule
SelfCheckGPT-reproduced & selfcheckgpt-original & 83.12 & 78.65 \\
SelfCheckGPT-reproduced & selfcheckgpt-enhanced & 82.36 & 76.71 \\
\bottomrule
\end{tabular}

